In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_3.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_3.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_3.csv')

In [3]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [4]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

30650
30650
30650


In [5]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[0].size())

tensor([0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.4882], device='cuda:0')
torch.Size([12289])


In [6]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([-0.0044, -0.0028], device='cuda:0')
torch.Size([2])


In [7]:
dataset = TensorDataset(X_tensor, y_tensor)
data_loader = DataLoader(dataset, batch_size=512, shuffle=False)

In [8]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=12289, hidden_size=512, hidden_size_1=512, output_size=2, num_layers=2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        self.linear = nn.Linear(hidden_size, hidden_size_1)
        self.fc2 = nn.Linear(hidden_size_1, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).cuda()
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).cuda()

        out, _ = self.lstm(x, (h0, c0))

        out = out[:, -1, :]

        out = self.linear(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.tanh(out)
        return out.squeeze(0)

In [9]:
model = LSTMModel().cuda()

In [10]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.000001)

# Обучение модели
num_epochs = 9
for epoch in range(num_epochs):
    for batch_x, batch_y in data_loader:
        optimizer.zero_grad()

        outputs = model(batch_x.unsqueeze(0))
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([512, 2])) that is different to the input size (torch.Size([2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([442, 2])) that is different to the input size (torch.Size([2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/9], Loss: 0.000718484
Epoch [2/9], Loss: 0.000527223
Epoch [3/9], Loss: 0.000395256
Epoch [4/9], Loss: 0.000304111
Epoch [5/9], Loss: 0.000239761
Epoch [6/9], Loss: 0.000193462
Epoch [7/9], Loss: 0.000158629
Epoch [8/9], Loss: 0.000132243
Epoch [9/9], Loss: 0.000111730


In [12]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_lstm_5_2.pth')